# WALINET simulator vs. baseline-free physics decoder

Dieses Notebook erzeugt metabolische Spektren mit dem echten WALINET-Simulator und übergibt **exakt dieselben gezogenen Parameter und dieselbe vorbereitete LCModel-Basis** an den neuen Decoder im Denoising-Projekt. Es gibt bewusst keine Baseline und keinen neuronalen Residualpfad.

Die Plots zeigen WALINET, Decoder und ihre komplexe Differenz. Bei korrekter Implementierung liegen die Kurven übereinander und das Residuum befindet sich im Bereich numerischer Rundungsfehler.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml


def find_project_root(start: Path, name: str) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if candidate.name == name and (candidate / 'src').is_dir():
            return candidate
        sibling = candidate / name
        if (sibling / 'src').is_dir():
            return sibling
    raise FileNotFoundError(f'Projekt {name!r} ausgehend von {start} nicht gefunden.')


DENOISING_ROOT = find_project_root(Path.cwd(), 'Denoising')
WALINET_ROOT = find_project_root(DENOISING_ROOT.parent, 'walinet')
for source_root in (DENOISING_ROOT / 'src', WALINET_ROOT / 'src'):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

from denoising.models.physics import BaselineFreeBasisDecoder, SpectralParameters
from walinet.config.build_simulation import build_simulation_config
from walinet.training_data.lcmodel_basis.acquisition import prepare_basis_for_acquisition
from walinet.training_data.metabolite_simulation import MetaboliteSimulator

print('Denoising:', DENOISING_ROOT)
print('WALINET:  ', WALINET_ROOT)

## Einstellungen

`FIELD_STRENGTH` kann auf `"7T"` oder `"3T"` gesetzt werden. Der Simulator läuft standardmäßig auf der gewählten GPU, fällt aber auch auf CPU zurück.

In [ ]:
FIELD_STRENGTH = '7T'  # '7T' oder '3T'
GPU_NUMBER = 0
BATCH_SIZE = 8
SEED = 42
EXAMPLE_INDEX = 0

DEVICE = torch.device(
    f'cuda:{GPU_NUMBER}' if torch.cuda.is_available() else 'cpu'
)
SIMULATION_CONFIG_PATH = (
    WALINET_ROOT / 'configs' / 'Simulation' / f'{FIELD_STRENGTH}_on_the_fly.yaml'
).resolve()

print('Device:            ', DEVICE)
print('Simulation config: ', SIMULATION_CONFIG_PATH)

## WALINET-Konfiguration und Basis laden

Die WALINET-Konfigurationsroutine löst alle relativen Pfade und konvertiert ppm-basierte Verteilungen in die intern verwendeten Einheiten. Anschließend wird die LCModel-Basis auf Bandbreite und Punktzahl der gewählten Akquisition vorbereitet.

In [ ]:
with SIMULATION_CONFIG_PATH.open('r', encoding='utf-8') as file:
    simulation_raw = yaml.safe_load(file)

simulation_config = build_simulation_config(
    simulation_raw,
    config_dir=SIMULATION_CONFIG_PATH.parent,
)
prepared_basis = prepare_basis_for_acquisition(
    simulation_config.basis.library,
    target_bandwidth=simulation_config.acquisition.bandwidth_hz,
    target_n_timepoints=simulation_config.acquisition.n_timepoints,
    dataset_name='clean_fid',
)

print(f'Basis components: {prepared_basis.n_metabolites}')
print(f'Timepoints:       {prepared_basis.n_timepoints}')
print(f'Bandwidth:        {prepared_basis.bandwidth:.6g} Hz')
print(f'Dwell time:       {prepared_basis.dwell_time:.9g} s')
print('Names:', ', '.join(prepared_basis.names))

## Ein gemeinsamer Parametersatz für beide Signalwege

WALINET zieht Konzentrationen, Frequenzshift, beide Phasen und die beiden Voigt-Komponenten. Der neue Decoder bekommt danach genau diese zurückgegebenen Tensoren. Es wird nichts erneut gezogen oder geschätzt.

In [ ]:
walinet_simulator = MetaboliteSimulator(
    prepared_basis=prepared_basis,
    config=simulation_config,
    device=DEVICE,
)
generator = torch.Generator(device=DEVICE).manual_seed(SEED)
walinet_result = walinet_simulator.simulate(
    batch_size=BATCH_SIZE,
    generator=generator,
)

basis_fids = torch.from_numpy(
    np.ascontiguousarray(prepared_basis.fids, dtype=np.complex64)
).to(DEVICE)
decoder = BaselineFreeBasisDecoder(
    basis_fids=basis_fids,
    dwell_time_seconds=prepared_basis.dwell_time,
).to(DEVICE)

shared_parameters = SpectralParameters(
    amplitudes=walinet_result.concentrations,
    frequency_shift_hz=walinet_result.frequency_shifts_hz,
    lorentzian_fwhm_hz=walinet_result.lorentzian_fwhm_hz,
    gaussian_fwhm_hz=walinet_result.gaussian_fwhm_hz,
    zero_order_phase_radians=walinet_result.zero_order_phases_radians,
    first_order_phase_rad_per_hz=walinet_result.first_order_phases_rad_per_hz,
)
decoder_fids = decoder.decode_fids(shared_parameters)
decoder_spectra = decoder.decode_spectra(shared_parameters)

fid_error = decoder_fids - walinet_result.clean_fids
spectrum_error = decoder_spectra - walinet_result.clean_spectra
print('Shapes:', tuple(decoder_spectra.shape))
print(f'Max |FID difference|:      {fid_error.abs().max().item():.9g}')
print(f'Max |spectrum difference|: {spectrum_error.abs().max().item():.9g}')
print(f'Mean |spectrum difference|:{spectrum_error.abs().mean().item():.9g}')
torch.testing.assert_close(decoder_fids, walinet_result.clean_fids)
torch.testing.assert_close(decoder_spectra, walinet_result.clean_spectra)
print('Numerical equivalence check: PASSED')

## Gezogene Parameter ansehen

In [ ]:
i = int(EXAMPLE_INDEX)
if not 0 <= i < BATCH_SIZE:
    raise IndexError(f'EXAMPLE_INDEX muss zwischen 0 und {BATCH_SIZE - 1} liegen.')

print(f'Example {i}')
print(f'Frequency shift: {walinet_result.frequency_shifts_hz[i].item():.5g} Hz')
print(f'Lorentz FWHM:    {walinet_result.lorentzian_fwhm_hz[i].item():.5g} Hz')
print(f'Gaussian FWHM:   {walinet_result.gaussian_fwhm_hz[i].item():.5g} Hz')
print(f'Phase 0:         {walinet_result.zero_order_phases_radians[i].item():.5g} rad')
print(f'Phase 1:         {walinet_result.first_order_phases_rad_per_hz[i].item():.5g} rad/Hz')
print('\nNon-zero amplitudes:')
for name, amplitude in zip(prepared_basis.names, walinet_result.concentrations[i].detach().cpu()):
    if amplitude.item() != 0:
        print(f'  {name:>12s}: {amplitude.item():.6g}')

## Visueller Vergleich eines Beispiels

Die oberen Reihen zeigen Real- und Imaginärteil. Unten stehen Betrag und komplexes Residuum. Die Frequenzachse ist bewusst in Hz angegeben, damit hier keine zusätzliche ppm- oder Darstellungs-Konvention in den Vergleich eingeht.

In [ ]:
wal_spec = walinet_result.clean_spectra[i].detach().cpu().numpy()
dec_spec = decoder_spectra[i].detach().cpu().numpy()
res_spec = dec_spec - wal_spec
frequency_hz = np.fft.fftshift(
    np.fft.fftfreq(prepared_basis.n_timepoints, d=prepared_basis.dwell_time)
)

fig, axes = plt.subplots(2, 2, figsize=(18, 10), constrained_layout=True)
for values, label, style in (
    (wal_spec, 'WALINET simulator', '-'),
    (dec_spec, 'Baseline-free decoder', '--'),
):
    axes[0, 0].plot(frequency_hz, values.real, style, lw=1.5, label=label)
    axes[0, 1].plot(frequency_hz, values.imag, style, lw=1.5, label=label)
    axes[1, 0].plot(frequency_hz, np.abs(values), style, lw=1.5, label=label)

axes[1, 1].plot(frequency_hz, res_spec.real, label='Residual real')
axes[1, 1].plot(frequency_hz, res_spec.imag, label='Residual imaginary', alpha=0.8)
axes[0, 0].set_title('Real spectrum')
axes[0, 1].set_title('Imaginary spectrum')
axes[1, 0].set_title('Magnitude spectrum')
axes[1, 1].set_title('Decoder − WALINET (numerical residual)')
for ax in axes.flat:
    ax.set_xlabel('Frequency offset [Hz]')
    ax.grid(alpha=0.2)
    ax.legend()
fig.suptitle(f'{FIELD_STRENGTH}, simulated example {i}', fontsize=18)
plt.show()

## Gesamten Batch vergleichen

Jede Zeile ist ein unabhängig simulierter Parametersatz. Links liegen beide Realteile übereinander, rechts ist das Residuum separat dargestellt.

In [ ]:
wal_batch = walinet_result.clean_spectra.detach().cpu().numpy()
dec_batch = decoder_spectra.detach().cpu().numpy()
fig, axes = plt.subplots(BATCH_SIZE, 2, figsize=(18, 3 * BATCH_SIZE), squeeze=False, constrained_layout=True)
for row in range(BATCH_SIZE):
    axes[row, 0].plot(frequency_hz, wal_batch[row].real, lw=1.3, label='WALINET')
    axes[row, 0].plot(frequency_hz, dec_batch[row].real, '--', lw=1.1, label='Decoder')
    residual = dec_batch[row] - wal_batch[row]
    axes[row, 1].plot(frequency_hz, residual.real, label='Real residual')
    axes[row, 1].plot(frequency_hz, residual.imag, label='Imag residual', alpha=0.8)
    axes[row, 0].set_ylabel(f'Example {row}')
    for ax in axes[row]:
        ax.grid(alpha=0.2)
        ax.legend(loc='upper right')
axes[0, 0].set_title('Real spectra: WALINET and decoder')
axes[0, 1].set_title('Complex numerical residual')
axes[-1, 0].set_xlabel('Frequency offset [Hz]')
axes[-1, 1].set_xlabel('Frequency offset [Hz]')
plt.show()